# Operaciones con vectores y matrices complejas

**Herramientas:** tipo `complex` de Python + `numpy`

---

Cada una de las 18 operaciones tiene la misma estructura:

1. **Definicion matematica** en una celda de texto.
2. **Implementacion propia** (con bucles) para entender *que* esta pasando.
3. **Verificacion con NumPy**, que es como se haria en la practica.


> Recordar: en Python la unidad imaginaria se escribe `j` y **siempre** va
> pegada a un numero: `3 + 4j` es valido, `3 + 4*j` no lo es (salvo que hayas
> definido `j` como variable).

## 0. Preparacion: el tipo `complex` en Python

In [ ]:
import numpy as np

# Para que las salidas se lean mejor
np.set_printoptions(precision=4, suppress=True)

print("Version de numpy:", np.__version__)

In [ ]:
# Tres formas equivalentes de crear un numero complejo
z1 = 3 + 4j
z2 = complex(1, -2)
z3 = np.complex128(-5 + 0j)

print("z1 =", z1, "| tipo:", type(z1))
print("z2 =", z2, "| tipo:", type(z2))
print("z3 =", z3, "| tipo:", type(z3))

In [ ]:
# Atributos y operaciones basicas de un complejo
z = 3 + 4j

print("Parte real      :", z.real)
print("Parte imaginaria:", z.imag)
print("Conjugado       :", z.conjugate())
print("Modulo |z|      :", abs(z))          # sqrt(3^2 + 4^2) = 5
print("z * conj(z)     :", z * z.conjugate())  # siempre real y no negativo

In [ ]:
# Vectores y matrices complejas con numpy.
# hay que pedir dtype=complex, si no numpy guarda solo la parte real.

v = np.array([1 + 2j, 3 - 1j, -2j], dtype=complex)
A = np.array([[1 + 1j, 2 - 1j],
              [0 + 3j, 4 + 0j]], dtype=complex)

print("Vector v:", v, "| shape:", v.shape)
print()
print("Matriz A:")
print(A)
print("shape:", A.shape, "| dtype:", A.dtype)

In [ ]:
def son_iguales(X, Y, tol=1e-9):
    """Compara dos arreglos complejos permitiendo error numerico.

    No usamos == porque las operaciones con flotantes acumulan errores
    diminutos (p. ej. 0.9999999999 en lugar de 1.0).
    """
    return np.allclose(np.array(X, dtype=complex),
                       np.array(Y, dtype=complex), atol=tol)

# Ejemplo del problema de comparar flotantes con ==
print(0.1 + 0.2 == 0.3)                 # False
print(son_iguales(0.1 + 0.2, 0.3))      # True

---
## 1. Adicion de vectores complejos

Si $u=(u_1,\dots,u_n)$ y $v=(v_1,\dots,v_n)$ son vectores de $\mathbb{C}^n$,
su suma se define **componente a componente**:

$$(u+v)_k = u_k + v_k, \qquad k = 1,\dots,n$$

Los dos vectores deben tener la **misma longitud**. La suma es conmutativa,
asociativa, tiene elemento neutro (el vector cero) y cada vector tiene inverso
aditivo: $(\mathbb{C}^n, +)$ es un grupo abeliano.

In [ ]:
def suma_vectores(u, v):
    """Suma dos vectores complejos componente a componente."""
    u = np.array(u, dtype=complex)
    v = np.array(v, dtype=complex)

    if u.shape != v.shape:
        raise ValueError("Los vectores deben tener la misma longitud")

    resultado = np.zeros(u.shape, dtype=complex)
    for k in range(len(u)):
        resultado[k] = u[k] + v[k]
    return resultado

In [ ]:
u = np.array([1 + 2j, 3 - 1j, 0 + 4j], dtype=complex)
v = np.array([2 - 1j, -1 + 5j, 3 + 0j], dtype=complex)

mio    = suma_vectores(u, v)
numpyy = u + v          # numpy suma elemento a elemento con el operador +

print("u        =", u)
print("v        =", v)
print("u + v    =", mio)
print("con numpy=", numpyy)
print("Coinciden:", son_iguales(mio, numpyy))

In [ ]:
# Comprobacion de las propiedades de grupo abeliano
w    = np.array([1 + 1j, 1 - 1j, 2 + 2j], dtype=complex)
cero = np.zeros(3, dtype=complex)

print("Conmutativa  u+v == v+u        :", son_iguales(u + v, v + u))
print("Asociativa   (u+v)+w == u+(v+w):", son_iguales((u + v) + w, u + (v + w)))
print("Neutro       u+0 == u          :", son_iguales(u + cero, u))

---
## 2. Inverso aditivo de un vector complejo

El inverso aditivo de $v$ es el unico vector $-v$ tal que $v + (-v) = \mathbf{0}$:

$$(-v)_k = -v_k$$

Negar un complejo cambia el signo de **ambas** partes:
$-(a+bi) = -a - bi$.

In [ ]:
def inverso_aditivo_vector(v):
    """Devuelve -v, el inverso aditivo del vector complejo v."""
    v = np.array(v, dtype=complex)
    resultado = np.zeros(v.shape, dtype=complex)
    for k in range(len(v)):
        resultado[k] = -v[k]
    return resultado

In [ ]:
v = np.array([1 + 2j, -3 + 1j, 0 - 4j], dtype=complex)

opuesto = inverso_aditivo_vector(v)

print("v          =", v)
print("-v         =", opuesto)
print("con numpy  =", -v)
print("v + (-v)   =", v + opuesto, "-> debe ser el vector cero")
print("Verificado :", son_iguales(v + opuesto, np.zeros(3)))

---
## 3. Multiplicacion de un escalar por un vector complejo

Para un escalar $c \in \mathbb{C}$ y un vector $v \in \mathbb{C}^n$:

$$(c\,v)_k = c \cdot v_k$$

Recuerda como se multiplican dos complejos:

$$(a+bi)(c+di) = (ac - bd) + (ad + bc)i$$

El escalar **tambien puede ser complejo**; ese es justamente el caso
interesante. Multiplicar por $i$, por ejemplo, equivale a rotar cada
componente $90^\circ$ en el plano complejo.

In [ ]:
def escalar_por_vector(c, v):
    """Multiplica el escalar complejo c por cada componente del vector v."""
    c = complex(c)
    v = np.array(v, dtype=complex)
    resultado = np.zeros(v.shape, dtype=complex)
    for k in range(len(v)):
        resultado[k] = c * v[k]
    return resultado

In [ ]:
c = 2 + 3j
v = np.array([1 + 1j, 2 - 2j, 0 + 1j], dtype=complex)

mio = escalar_por_vector(c, v)

print("c        =", c)
print("v        =", v)
print("c*v      =", mio)
print("con numpy=", c * v)
print("Coinciden:", son_iguales(mio, c * v))
print()
print("Rotacion: i*v =", escalar_por_vector(1j, v))

In [ ]:
# Propiedades del producto por escalar
c1, c2 = 2 + 1j, 1 - 3j
u = np.array([1 + 0j, 0 + 2j, 3 - 1j], dtype=complex)

print("Distributiva  c(u+v) == cu+cv :", son_iguales(c1*(u + v), c1*u + c1*v))
print("Distributiva (c1+c2)v==c1v+c2v:", son_iguales((c1 + c2)*v, c1*v + c2*v))
print("Asociativa   c1(c2 v)==(c1c2)v:", son_iguales(c1*(c2*v), (c1*c2)*v))
print("Neutro        1*v == v        :", son_iguales(1*v, v))

---
## 4. Adicion de matrices complejas

Igual que con vectores, la suma es **entrada a entrada**. Si $A$ y $B$ son
matrices $m \times n$ sobre $\mathbb{C}$:

$$(A+B)_{jk} = A_{jk} + B_{jk}$$

Ambas matrices deben tener **exactamente la misma forma** ($m \times n$).

In [ ]:
def suma_matrices(A, B):
    """Suma dos matrices complejas entrada a entrada."""
    A = np.array(A, dtype=complex)
    B = np.array(B, dtype=complex)

    if A.shape != B.shape:
        raise ValueError("Las matrices deben tener la misma forma")

    filas, columnas = A.shape
    resultado = np.zeros((filas, columnas), dtype=complex)
    for j in range(filas):
        for k in range(columnas):
            resultado[j, k] = A[j, k] + B[j, k]
    return resultado

In [ ]:
A = np.array([[1 + 1j, 2 - 1j],
              [0 + 3j, 4 + 0j]], dtype=complex)

B = np.array([[2 + 0j, -1 + 1j],
              [1 - 2j,  0 + 5j]], dtype=complex)

mio = suma_matrices(A, B)

print("A:\n", A, sep="")
print("\nB:\n", B, sep="")
print("\nA + B (implementacion propia):\n", mio, sep="")
print("\nA + B (numpy):\n", A + B, sep="")
print("\nCoinciden:", son_iguales(mio, A + B))

---
## 5. Inversa aditiva de una matriz compleja

$$(-A)_{jk} = -A_{jk}$$

Cumple $A + (-A) = \mathbf{0}_{m\times n}$ (la matriz nula).

> No confundir con la **inversa multiplicativa** $A^{-1}$, que solo existe
> para matrices cuadradas con determinante distinto de cero y se calcula con
> `np.linalg.inv`.

In [ ]:
def inverso_aditivo_matriz(A):
    """Devuelve -A, la inversa aditiva de la matriz compleja A."""
    A = np.array(A, dtype=complex)
    filas, columnas = A.shape
    resultado = np.zeros((filas, columnas), dtype=complex)
    for j in range(filas):
        for k in range(columnas):
            resultado[j, k] = -A[j, k]
    return resultado

In [ ]:
opuesta = inverso_aditivo_matriz(A)

print("A:\n", A, sep="")
print("\n-A:\n", opuesta, sep="")
print("\nA + (-A):\n", A + opuesta, sep="")
print("\nEs la matriz nula:", son_iguales(A + opuesta, np.zeros((2, 2))))

---
## 6. Multiplicacion de un escalar por una matriz compleja

$$(cA)_{jk} = c \cdot A_{jk}$$

Cada entrada se multiplica por el mismo escalar $c \in \mathbb{C}$.

In [ ]:
def escalar_por_matriz(c, A):
    """Multiplica el escalar complejo c por cada entrada de la matriz A."""
    c = complex(c)
    A = np.array(A, dtype=complex)
    filas, columnas = A.shape
    resultado = np.zeros((filas, columnas), dtype=complex)
    for j in range(filas):
        for k in range(columnas):
            resultado[j, k] = c * A[j, k]
    return resultado

In [ ]:
c = 1 + 2j
mio = escalar_por_matriz(c, A)

print("c =", c)
print("\nc*A (propia):\n", mio, sep="")
print("\nc*A (numpy):\n", c * A, sep="")
print("\nCoinciden:", son_iguales(mio, c * A))

---
## 7. Transpuesta de una matriz o vector

La transpuesta intercambia filas por columnas:

$$(A^{T})_{jk} = A_{kj}$$

Si $A$ es $m \times n$, entonces $A^{T}$ es $n \times m$.

**Detalle importante con NumPy:** un arreglo de una dimension
(`shape == (n,)`) **no cambia** al aplicarle `.T`, porque numpy no distingue
entre vector fila y vector columna. Para trabajar con vectores columna de
forma explicita hay que darles forma $n \times 1$ con `reshape(-1, 1)`.

In [ ]:
def transpuesta(M):
    """Transpuesta de una matriz. Un vector 1-D se trata como vector columna."""
    M = np.array(M, dtype=complex)

    if M.ndim == 1:                    # vector -> lo vemos como columna n x 1
        M = M.reshape(len(M), 1)

    filas, columnas = M.shape
    resultado = np.zeros((columnas, filas), dtype=complex)
    for j in range(filas):
        for k in range(columnas):
            resultado[k, j] = M[j, k]
    return resultado

In [ ]:
M = np.array([[1 + 1j, 2 - 1j, 3 + 0j],
              [0 + 3j, 4 + 0j, 5 - 2j]], dtype=complex)

print("M (2x3):\n", M, sep="")
print("\nM^T (3x2), propia:\n", transpuesta(M), sep="")
print("\nM^T con numpy:\n", M.T, sep="")
print("\nCoinciden:", son_iguales(transpuesta(M), M.T))

In [ ]:
# El caso de los vectores
v = np.array([1 + 2j, 3 - 1j, 0 + 4j], dtype=complex)

print("v          :", v, " shape:", v.shape)
print("v.T (numpy):", v.T, " shape:", v.T.shape, " <- no cambio nada")
print()
print("Vector columna explicito (3x1):")
print(transpuesta(v))
print("shape:", transpuesta(v).shape)

---
## 8. Conjugada de una matriz o vector

Se conjuga **cada entrada**:

$$(\bar{A})_{jk} = \overline{A_{jk}}, \qquad \overline{a+bi} = a-bi$$

Geometricamente, conjugar refleja cada entrada respecto al eje real.
La forma de la matriz no cambia.

In [ ]:
def conjugada(M):
    """Conjuga cada entrada de una matriz o vector complejo."""
    M = np.array(M, dtype=complex)
    resultado = np.zeros(M.shape, dtype=complex)

    if M.ndim == 1:
        for k in range(M.shape[0]):
            resultado[k] = M[k].conjugate()
    else:
        filas, columnas = M.shape
        for j in range(filas):
            for k in range(columnas):
                resultado[j, k] = M[j, k].conjugate()
    return resultado

In [ ]:
print("M:\n", M, sep="")
print("\nconj(M) propia:\n", conjugada(M), sep="")
print("\nconj(M) numpy:\n", np.conjugate(M), sep="")
print("\nCoinciden:", son_iguales(conjugada(M), np.conjugate(M)))
print()
print("conj(v):", conjugada(v), " | numpy:", v.conj())
print("Doble conjugada devuelve el original:", son_iguales(conjugada(conjugada(M)), M))

---
## 9. Adjunta (daga) de una matriz o vector

La adjunta, tambien llamada **transpuesta conjugada** o **daga**, combina las
dos operaciones anteriores:

$$A^{\dagger} = \overline{A^{T}} = \left(\overline{A}\right)^{T},
\qquad (A^{\dagger})_{jk} = \overline{A_{kj}}$$

El orden no importa: transponer y luego conjugar da lo mismo que conjugar y
luego transponer.

Propiedades utiles:

- $(A^{\dagger})^{\dagger} = A$
- $(A+B)^{\dagger} = A^{\dagger} + B^{\dagger}$
- $(AB)^{\dagger} = B^{\dagger}A^{\dagger}$  (se invierte el orden)
- $(cA)^{\dagger} = \bar{c}\,A^{\dagger}$

In [ ]:
def adjunta(M):
    """Adjunta (daga) de M: transpuesta conjugada."""
    return transpuesta(conjugada(M))

In [ ]:
print("M (2x3):\n", M, sep="")
print("\nM^dagger (3x2) propia:\n", adjunta(M), sep="")
print("\nM^dagger con numpy (M.conj().T):\n", M.conj().T, sep="")
print("\nCoinciden:", son_iguales(adjunta(M), M.conj().T))
print()
print("(A^dagger)^dagger == A :", son_iguales(adjunta(adjunta(A)), A))
print("(A+B)^d == A^d + B^d   :", son_iguales(adjunta(A + B), adjunta(A) + adjunta(B)))
print("(cA)^d == conj(c) A^d  :", son_iguales(adjunta(c * A), np.conjugate(c) * adjunta(A)))

---
## 10. Producto de dos matrices

Si $A$ es $m \times n$ y $B$ es $n \times p$, el producto $AB$ es $m \times p$:

$$(AB)_{jk} = \sum_{l=1}^{n} A_{jl}\,B_{lk}$$

La condicion de compatibilidad es que el **numero de columnas de $A$ sea igual
al numero de filas de $B$**.

El producto de matrices **no es conmutativo**: en general $AB \neq BA$.

In [ ]:
def producto_matrices(A, B):
    """Producto matricial A*B usando la definicion con sumatorias."""
    A = np.array(A, dtype=complex)
    B = np.array(B, dtype=complex)

    if A.shape[1] != B.shape[0]:
        raise ValueError(
            f"Dimensiones incompatibles: {A.shape} x {B.shape}. "
            "Las columnas de A deben coincidir con las filas de B."
        )

    m, n = A.shape
    _, p = B.shape
    resultado = np.zeros((m, p), dtype=complex)

    for j in range(m):
        for k in range(p):
            suma = 0 + 0j
            for l in range(n):
                suma += A[j, l] * B[l, k]
            resultado[j, k] = suma
    return resultado

In [ ]:
P = np.array([[1 + 1j, 2 + 0j],
              [0 - 1j, 3 + 2j]], dtype=complex)

Q = np.array([[2 + 0j, 1 - 1j],
              [1 + 1j, 0 + 2j]], dtype=complex)

mio = producto_matrices(P, Q)

print("P:\n", P, sep="")
print("\nQ:\n", Q, sep="")
print("\nP*Q (propia):\n", mio, sep="")
print("\nP*Q (numpy, operador @):\n", P @ Q, sep="")
print("\nCoinciden:", son_iguales(mio, P @ Q))
print()
print("Q*P:\n", Q @ P, sep="")
print("\nEs conmutativo P*Q == Q*P?", son_iguales(P @ Q, Q @ P))

In [ ]:
# CUIDADO: el operador * en numpy NO es el producto matricial,
# es la multiplicacion elemento a elemento (producto de Hadamard).
print("P @ P (producto matricial):\n", P @ P, sep="")
print("\nP * P (elemento a elemento, NO es lo que queremos):\n", P * P, sep="")

In [ ]:
# Producto entre matrices de tamanos distintos pero compatibles
R = np.array([[1 + 0j, 0 + 1j, 2 - 1j],
              [3 + 1j, 1 - 1j, 0 + 0j]], dtype=complex)     # 2x3
S = np.array([[1 + 1j],
              [0 + 2j],
              [1 - 1j]], dtype=complex)                      # 3x1

print("R:", R.shape, " S:", S.shape, " -> R*S:", (R @ S).shape)
print("\nR*S:\n", producto_matrices(R, S), sep="")

# El producto en el otro sentido no esta definido
try:
    producto_matrices(S, R.T)
except ValueError as e:
    print("\nError esperado:", e)

---
## 11. Accion de una matriz sobre un vector

La "accion" de una matriz $A$ ($m \times n$) sobre un vector $v \in \mathbb{C}^n$
produce un vector de $\mathbb{C}^m$:

$$(Av)_j = \sum_{k=1}^{n} A_{jk}\,v_k$$

Es un caso particular del producto de matrices, tomando $v$ como matriz
$n \times 1$. Toda matriz define asi una **transformacion lineal**:

$$A(u+v) = Au + Av, \qquad A(cv) = c\,(Av)$$

In [ ]:
def accion(A, v):
    """Aplica la matriz A al vector v y devuelve el vector resultante (1-D)."""
    A = np.array(A, dtype=complex)
    v = np.array(v, dtype=complex)

    if A.shape[1] != v.shape[0]:
        raise ValueError(
            f"No se puede aplicar una matriz {A.shape} a un vector de "
            f"longitud {v.shape[0]}"
        )

    filas = A.shape[0]
    resultado = np.zeros(filas, dtype=complex)
    for j in range(filas):
        suma = 0 + 0j
        for k in range(A.shape[1]):
            suma += A[j, k] * v[k]
        resultado[j] = suma
    return resultado

In [ ]:
x = np.array([1 + 1j, 2 - 1j], dtype=complex)

mio = accion(P, x)

print("P:\n", P, sep="")
print("\nx =", x)
print("P x (propia) =", mio)
print("P x (numpy)  =", P @ x)
print("Coinciden    :", son_iguales(mio, P @ x))

In [ ]:
# Linealidad de la accion
y  = np.array([0 + 2j, 1 - 3j], dtype=complex)
cc = 2 - 1j

print("A(x+y) == Ax + Ay :", son_iguales(accion(P, x + y), accion(P, x) + accion(P, y)))
print("A(cx)  == c(Ax)   :", son_iguales(accion(P, cc * x), cc * accion(P, x)))

---
## 12. Producto interno de dos vectores

En $\mathbb{C}^n$ el producto interno **no** es la simple suma de productos:
hay que conjugar uno de los dos vectores. Usando la convencion de la fisica
(conjugar el **primer** argumento):

$$\langle u, v \rangle = u^{\dagger} v = \sum_{k=1}^{n} \overline{u_k}\,v_k$$

Propiedades:

- $\langle u,v \rangle = \overline{\langle v,u \rangle}$ (simetria hermitiana,
  **no** es simetrico)
- Lineal en el segundo argumento, **anti**lineal en el primero:
  $\langle cu, v\rangle = \bar{c}\langle u,v\rangle$ y
  $\langle u, cv\rangle = c\langle u,v\rangle$
- $\langle v,v \rangle \geq 0$ y es **siempre un numero real**, incluso si el
  vector es complejo. Vale $0$ solo si $v = \mathbf{0}$.

> En NumPy: `np.vdot(u, v)` conjuga el primer argumento (es lo que queremos).
> `np.dot(u, v)` **no conjuga nada**, asi que no sirve como producto interno
> complejo. Es un error muy comun.

In [ ]:
def producto_interno(u, v):
    """Producto interno <u,v> = suma de conj(u_k)*v_k."""
    u = np.array(u, dtype=complex)
    v = np.array(v, dtype=complex)

    if u.shape != v.shape:
        raise ValueError("Los vectores deben tener la misma longitud")

    total = 0 + 0j
    for k in range(len(u)):
        total += u[k].conjugate() * v[k]
    return total

In [ ]:
u = np.array([1 + 2j, 3 - 1j], dtype=complex)
v = np.array([2 - 1j, 0 + 4j], dtype=complex)

mio = producto_interno(u, v)

print("u =", u)
print("v =", v)
print()
print("<u,v> propia      :", mio)
print("<u,v> np.vdot(u,v):", np.vdot(u, v))
print("Coinciden         :", son_iguales(mio, np.vdot(u, v)))
print()
print("np.dot(u,v) SIN conjugar (incorrecto como prod. interno):", np.dot(u, v))

In [ ]:
print("<u,v> == conj(<v,u>)  :", son_iguales(producto_interno(u, v),
                                             np.conjugate(producto_interno(v, u))))
print("<cu,v> == conj(c)<u,v>:", son_iguales(producto_interno(cc * u, v),
                                             np.conjugate(cc) * producto_interno(u, v)))
print("<u,cv> == c<u,v>      :", son_iguales(producto_interno(u, cc * v),
                                             cc * producto_interno(u, v)))
print()
print("<v,v> =", producto_interno(v, v), "-> parte imaginaria nula, valor real y positivo")

---
## 13. Norma de un vector

La norma (o longitud) se define a partir del producto interno:

$$\|v\| = \sqrt{\langle v, v \rangle} = \sqrt{\sum_{k=1}^{n} |v_k|^2}
        = \sqrt{\sum_{k=1}^{n} \left( a_k^2 + b_k^2 \right)}$$

donde $v_k = a_k + b_k i$. Siempre es un **numero real no negativo**.

Propiedades: $\|cv\| = |c|\,\|v\|$ y la desigualdad triangular
$\|u+v\| \leq \|u\| + \|v\|$.

In [ ]:
def norma(v):
    """Norma euclidiana de un vector complejo. Devuelve un float real."""
    v = np.array(v, dtype=complex)
    total = 0.0
    for k in range(len(v)):
        # |v_k|^2 = real^2 + imag^2
        total += v[k].real**2 + v[k].imag**2
    return np.sqrt(total)

In [ ]:
v = np.array([3 + 4j, 0 + 0j], dtype=complex)   # |3+4i| = 5

print("v         =", v)
print("||v|| propia :", norma(v))
print("||v|| numpy  :", np.linalg.norm(v))
print()

w = np.array([1 + 1j, 2 - 1j, 0 + 3j], dtype=complex)
print("w        =", w)
print("||w|| propia:", norma(w))
print("||w|| numpy :", np.linalg.norm(w))
print("Coinciden   :", son_iguales(norma(w), np.linalg.norm(w)))

In [ ]:
print("||cv|| == |c| ||v||       :", son_iguales(norma(cc * w), abs(cc) * norma(w)))
print("Desigualdad triangular    :", norma(u + v) <= norma(u) + norma(v) + 1e-9)
print()

# Normalizar: obtener un vector unitario en la misma direccion
w_unitario = w / norma(w)
print("w normalizado:", w_unitario)
print("Su norma     :", norma(w_unitario))

---
## 14. Distancia entre dos vectores

La distancia es la norma de la diferencia:

$$d(u,v) = \|u - v\| = \sqrt{\sum_{k=1}^{n} |u_k - v_k|^2}$$

Es una **metrica**: $d(u,v) \geq 0$, $d(u,v)=0 \iff u=v$,
$d(u,v)=d(v,u)$ y $d(u,w) \leq d(u,v)+d(v,w)$.

In [ ]:
def distancia(u, v):
    """Distancia euclidiana entre dos vectores complejos."""
    u = np.array(u, dtype=complex)
    v = np.array(v, dtype=complex)

    if u.shape != v.shape:
        raise ValueError("Los vectores deben tener la misma longitud")

    return norma(u - v)

In [ ]:
u = np.array([1 + 2j, 3 - 1j], dtype=complex)
v = np.array([2 - 1j, 0 + 4j], dtype=complex)

print("u =", u)
print("v =", v)
print()
print("d(u,v) propia:", distancia(u, v))
print("d(u,v) numpy :", np.linalg.norm(u - v))
print()
print("Simetrica  d(u,v)==d(v,u):", son_iguales(distancia(u, v), distancia(v, u)))
print("d(u,u) == 0             :", son_iguales(distancia(u, u), 0))

---
## 15. Valores y vectores propios de una matriz

Un escalar $\lambda \in \mathbb{C}$ es **valor propio** de la matriz cuadrada
$A$ si existe un vector no nulo $v$ tal que

$$A v = \lambda v$$

Ese $v$ es un **vector propio** asociado a $\lambda$. Los valores propios son
las raices del polinomio caracteristico $\det(A - \lambda I) = 0$.

Sobre $\mathbb{C}$ toda matriz $n \times n$ tiene exactamente $n$ valores
propios contando multiplicidades (teorema fundamental del algebra).

**En NumPy:** `np.linalg.eig(A)` devuelve `(valores, vectores)` donde los
vectores propios son las **columnas** de la segunda matriz, ya normalizados.
Para matrices hermitianas conviene usar `np.linalg.eigh`, que es mas estable
numericamente.

In [ ]:
def valores_y_vectores_propios(A):
    """Devuelve (valores, vectores) de la matriz cuadrada A.

    Los vectores propios vienen como columnas de la matriz devuelta.
    """
    A = np.array(A, dtype=complex)
    if A.shape[0] != A.shape[1]:
        raise ValueError("La matriz debe ser cuadrada")
    return np.linalg.eig(A)

In [ ]:
A = np.array([[2 + 0j, 1 - 1j],
              [1 + 1j, 3 + 0j]], dtype=complex)

valores, vectores = valores_y_vectores_propios(A)

print("A:\n", A, sep="")
print("\nValores propios:", valores)
print("\nVectores propios (uno por columna):\n", vectores, sep="")

In [ ]:
# Verificacion: para cada par, A v debe ser igual a lambda v
for i in range(len(valores)):
    lam = valores[i]
    vec = vectores[:, i]          # columna i-esima
    print(f"--- Valor propio {i}: lambda = {lam:.4f}")
    print("  v      =", np.round(vec, 4))
    print("  A v    =", np.round(A @ vec, 4))
    print("  lambda v =", np.round(lam * vec, 4))
    print("  Se cumple A v = lambda v:", son_iguales(A @ vec, lam * vec))
    print("  Norma del vector propio:", round(float(norma(vec)), 6))

In [ ]:
# Un caso donde los valores propios son genuinamente complejos:
# la matriz de rotacion de 90 grados
Rot = np.array([[0, -1],
                [1,  0]], dtype=complex)

val_rot, vec_rot = np.linalg.eig(Rot)
print("Matriz de rotacion:\n", Rot.real.astype(int), sep="")
print("\nValores propios:", val_rot, " -> son i y -i, sin parte real")
print("\nTraza de A  =", np.trace(A), " | suma de valores propios =", np.sum(valores))
print("det(A)      =", np.linalg.det(A), " | producto de valores propios =", np.prod(valores))

---
## 16. Revisar si una matriz es unitaria

Una matriz cuadrada $U$ es **unitaria** si

$$U^{\dagger} U = U U^{\dagger} = I$$

Equivalentemente, $U^{-1} = U^{\dagger}$. Sus columnas (y sus filas) forman una
base ortonormal de $\mathbb{C}^n$.

Las matrices unitarias **preservan el producto interno y la norma**:
$\langle Uu, Uv\rangle = \langle u,v\rangle$, por eso son las que describen la
evolucion de los estados en computacion cuantica. Todos sus valores propios
tienen modulo $1$.

> Al programar la verificacion **nunca** se compara con `==`: hay que usar una
> tolerancia, porque el producto acumula errores de punto flotante.

In [ ]:
def es_unitaria(U, tol=1e-9):
    """Devuelve True si U es unitaria (U^dagger U = U U^dagger = I)."""
    U = np.array(U, dtype=complex)

    if U.ndim != 2 or U.shape[0] != U.shape[1]:
        return False

    n = U.shape[0]
    I = np.eye(n, dtype=complex)
    Ud = adjunta(U)

    return son_iguales(Ud @ U, I, tol) and son_iguales(U @ Ud, I, tol)

In [ ]:
# Compuerta de Hadamard: unitaria
H = (1 / np.sqrt(2)) * np.array([[1,  1],
                                 [1, -1]], dtype=complex)

# Compuerta de fase S: unitaria
S = np.array([[1, 0],
              [0, 1j]], dtype=complex)

# Una matriz cualquiera: no unitaria
N = np.array([[1 + 1j, 2],
              [0,      3]], dtype=complex)

print("H es unitaria:", es_unitaria(H))
print("S es unitaria:", es_unitaria(S))
print("N es unitaria:", es_unitaria(N))
print()
print("H^dagger H:\n", np.round(adjunta(H) @ H, 10), sep="")

In [ ]:
# Las unitarias preservan la norma y el producto interno
z1 = np.array([1 + 1j, 2 - 1j], dtype=complex)
z2 = np.array([0 + 2j, 1 + 0j], dtype=complex)

print("||z1||    =", norma(z1))
print("||H z1||  =", norma(H @ z1), " <- igual")
print()
print("<z1,z2>     =", producto_interno(z1, z2))
print("<Hz1,Hz2>   =", producto_interno(H @ z1, H @ z2))
print()
print("Modulos de los valores propios de H:", np.abs(np.linalg.eigvals(H)))

---
## 17. Revisar si una matriz es Hermitiana

Una matriz cuadrada $A$ es **hermitiana** (o autoadjunta) si

$$A^{\dagger} = A, \qquad \text{es decir} \qquad A_{jk} = \overline{A_{kj}}$$

Consecuencias inmediatas:

- La **diagonal** debe ser real ($A_{jj} = \overline{A_{jj}}$).
- Los elementos simetricos respecto a la diagonal son conjugados entre si.
- Todos sus **valores propios son reales** y los vectores propios asociados a
  valores propios distintos son ortogonales.

Es el analogo complejo de una matriz simetrica real. En mecanica cuantica los
observables se representan con operadores hermitianos, precisamente porque las
mediciones deben dar numeros reales.

In [ ]:
def es_hermitiana(A, tol=1e-9):
    """Devuelve True si A es hermitiana (A^dagger = A)."""
    A = np.array(A, dtype=complex)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        return False

    return son_iguales(A, adjunta(A), tol)

In [ ]:
Herm = np.array([[3 + 0j, 2 - 1j],
                 [2 + 1j, 5 + 0j]], dtype=complex)

NoHerm = np.array([[3 + 1j, 2 - 1j],     # diagonal con parte imaginaria
                   [2 + 1j, 5 + 0j]], dtype=complex)

print("Herm es hermitiana  :", es_hermitiana(Herm))
print("NoHerm es hermitiana:", es_hermitiana(NoHerm))
print("H (Hadamard) es hermitiana:", es_hermitiana(H))
print()
print("Herm^dagger:\n", adjunta(Herm), sep="")

In [ ]:
# Los valores propios de una hermitiana son reales
val = np.linalg.eigvals(Herm)
print("Valores propios de Herm:", val)
print("Partes imaginarias     :", np.round(val.imag, 12), " -> todas nulas")
print()

# Las matrices de Pauli: hermitianas Y unitarias a la vez
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

for nombre, Pauli in [("X", X), ("Y", Y), ("Z", Z)]:
    print(f"Pauli {nombre}: hermitiana={es_hermitiana(Pauli)}, unitaria={es_unitaria(Pauli)}")

---
## 18. Producto tensor (producto de Kronecker)

Si $A$ es $m \times n$ y $B$ es $p \times q$, el producto tensor
$A \otimes B$ es una matriz $mp \times nq$ formada por bloques:

$$A \otimes B =
\begin{pmatrix}
A_{11}B & A_{12}B & \cdots & A_{1n}B \\
A_{21}B & A_{22}B & \cdots & A_{2n}B \\
\vdots  & \vdots  & \ddots & \vdots \\
A_{m1}B & A_{m2}B & \cdots & A_{mn}B
\end{pmatrix}$$

Cada entrada de $A$ se reemplaza por una copia completa de $B$ escalada por esa
entrada. **No es conmutativo**: $A \otimes B \neq B \otimes A$ en general.

Propiedad clave (regla del producto mixto):
$(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$

Para vectores, $u \otimes v$ tiene longitud $mn$ y es la forma de describir
sistemas compuestos: dos qubits viven en $\mathbb{C}^2 \otimes \mathbb{C}^2
= \mathbb{C}^4$.

In [ ]:
def producto_tensor(A, B):
    """Producto tensor (Kronecker) de dos matrices o vectores complejos."""
    A = np.array(A, dtype=complex)
    B = np.array(B, dtype=complex)

    # Los vectores 1-D se tratan como columnas
    vector_entrada = (A.ndim == 1 and B.ndim == 1)
    if A.ndim == 1:
        A = A.reshape(-1, 1)
    if B.ndim == 1:
        B = B.reshape(-1, 1)

    m, n = A.shape
    p, q = B.shape
    resultado = np.zeros((m * p, n * q), dtype=complex)

    for j in range(m):
        for k in range(n):
            # bloque (j,k) = A[j,k] * B
            for r in range(p):
                for s in range(q):
                    resultado[j * p + r, k * q + s] = A[j, k] * B[r, s]

    if vector_entrada:
        return resultado.reshape(-1)     # devolvemos un vector 1-D
    return resultado

In [ ]:
A2 = np.array([[1 + 0j, 2 - 1j],
               [0 + 1j, 3 + 0j]], dtype=complex)

B2 = np.array([[0 + 1j, 1 + 0j],
               [1 + 1j, 0 + 0j]], dtype=complex)

mio = producto_tensor(A2, B2)

print("A2:\n", A2, sep="")
print("\nB2:\n", B2, sep="")
print("\nA2 (x) B2  ->  shape", mio.shape, ":\n", mio, sep="")
print("\nnp.kron(A2,B2):\n", np.kron(A2, B2), sep="")
print("\nCoinciden:", son_iguales(mio, np.kron(A2, B2)))
print("Es conmutativo?", son_iguales(np.kron(A2, B2), np.kron(B2, A2)))

In [ ]:
# Producto tensor de vectores: estados de dos qubits
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

print("|0> (x) |0> =", producto_tensor(ket0, ket0).real.astype(int))
print("|0> (x) |1> =", producto_tensor(ket0, ket1).real.astype(int))
print("|1> (x) |0> =", producto_tensor(ket1, ket0).real.astype(int))
print("|1> (x) |1> =", producto_tensor(ket1, ket1).real.astype(int))
print()
print("Coincide con np.kron:", son_iguales(producto_tensor(ket0, ket1),
                                           np.kron(ket0, ket1)))

In [ ]:
# Regla del producto mixto y comportamiento con las propiedades ya vistas
C2 = np.array([[1, 1], [0, 1]], dtype=complex)
D2 = np.array([[2, 0], [1, 1j]], dtype=complex)

izq = producto_tensor(A2, B2) @ producto_tensor(C2, D2)
der = producto_tensor(A2 @ C2, B2 @ D2)
print("(A(x)B)(C(x)D) == (AC)(x)(BD):", son_iguales(izq, der))

print("H (x) H es unitaria          :", es_unitaria(np.kron(H, H)))
print("(A(x)B)^dagger == A^d (x) B^d:",
      son_iguales(adjunta(np.kron(A2, B2)), np.kron(adjunta(A2), adjunta(B2))))

---
# Pruebas automaticas

Todas las funciones implementadas se contrastan contra la version equivalente
de NumPy. Si alguna falla, la celda lanza un `AssertionError` indicando cual.

In [ ]:
def correr_pruebas():
    fallos = []

    def chequear(nombre, condicion):
        if condicion:
            print(f"  OK    {nombre}")
        else:
            print(f"  FALLA {nombre}")
            fallos.append(nombre)

    # Datos de prueba
    u = np.array([1 + 2j, 3 - 1j, 0 + 4j], dtype=complex)
    v = np.array([2 - 1j, -1 + 5j, 3 + 0j], dtype=complex)
    A = np.array([[1 + 1j, 2 - 1j], [0 + 3j, 4 + 0j]], dtype=complex)
    B = np.array([[2 + 0j, -1 + 1j], [1 - 2j, 0 + 5j]], dtype=complex)
    M = np.array([[1 + 1j, 2 - 1j, 3 + 0j], [0 + 3j, 4 + 0j, 5 - 2j]], dtype=complex)
    c = 2 + 3j
    H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
    Herm = np.array([[3 + 0j, 2 - 1j], [2 + 1j, 5 + 0j]], dtype=complex)

    print("Ejecutando pruebas...\n")
    chequear("01 suma de vectores",       son_iguales(suma_vectores(u, v), u + v))
    chequear("02 inverso aditivo vector", son_iguales(inverso_aditivo_vector(u), -u))
    chequear("03 escalar por vector",     son_iguales(escalar_por_vector(c, u), c * u))
    chequear("04 suma de matrices",       son_iguales(suma_matrices(A, B), A + B))
    chequear("05 inverso aditivo matriz", son_iguales(inverso_aditivo_matriz(A), -A))
    chequear("06 escalar por matriz",     son_iguales(escalar_por_matriz(c, A), c * A))
    chequear("07 transpuesta",            son_iguales(transpuesta(M), M.T))
    chequear("08 conjugada",              son_iguales(conjugada(M), M.conj()))
    chequear("09 adjunta",                son_iguales(adjunta(M), M.conj().T))
    chequear("10 producto de matrices",   son_iguales(producto_matrices(A, B), A @ B))
    chequear("11 accion sobre vector",    son_iguales(accion(A, np.array([1 + 1j, 2 - 1j])),
                                                      A @ np.array([1 + 1j, 2 - 1j])))
    chequear("12 producto interno",       son_iguales(producto_interno(u, v), np.vdot(u, v)))
    chequear("13 norma",                  son_iguales(norma(u), np.linalg.norm(u)))
    chequear("14 distancia",              son_iguales(distancia(u, v), np.linalg.norm(u - v)))

    val, vec = valores_y_vectores_propios(Herm)
    chequear("15 valores/vectores propios",
             all(son_iguales(Herm @ vec[:, i], val[i] * vec[:, i]) for i in range(len(val))))

    chequear("16 unitaria (positivo)",    es_unitaria(H) is True)
    chequear("16 unitaria (negativo)",    es_unitaria(A) is False)
    chequear("17 hermitiana (positivo)",  es_hermitiana(Herm) is True)
    chequear("17 hermitiana (negativo)",  es_hermitiana(A) is False)
    chequear("18 producto tensor",        son_iguales(producto_tensor(A, B), np.kron(A, B)))
    chequear("18 producto tensor vectores",
             son_iguales(producto_tensor(u, v), np.kron(u, v)))

    print()
    assert not fallos, f"Pruebas fallidas: {fallos}"
    print("Todas las pruebas pasaron correctamente.")


correr_pruebas()

---
# Resumen: equivalencias en NumPy

| # | Operacion | NumPy |
|---|-----------|-------|
| 1 | Suma de vectores | `u + v` |
| 2 | Inverso aditivo | `-v` |
| 3 | Escalar por vector | `c * v` |
| 4 | Suma de matrices | `A + B` |
| 5 | Inversa aditiva de matriz | `-A` |
| 6 | Escalar por matriz | `c * A` |
| 7 | Transpuesta | `A.T` |
| 8 | Conjugada | `np.conjugate(A)` o `A.conj()` |
| 9 | Adjunta (daga) | `A.conj().T` |
| 10 | Producto de matrices | `A @ B` |
| 11 | Accion sobre un vector | `A @ v` |
| 12 | Producto interno | `np.vdot(u, v)` |
| 13 | Norma | `np.linalg.norm(v)` |
| 14 | Distancia | `np.linalg.norm(u - v)` |
| 15 | Valores/vectores propios | `np.linalg.eig(A)` |
| 16 | Es unitaria | `np.allclose(A.conj().T @ A, np.eye(n))` |
| 17 | Es hermitiana | `np.allclose(A, A.conj().T)` |
| 18 | Producto tensor | `np.kron(A, B)` |

### Errores frecuentes que conviene recordar

- Olvidar `dtype=complex` al crear el arreglo: numpy descarta la parte imaginaria.
- Usar `*` esperando el producto matricial (es elemento a elemento; el producto
  matricial es `@`).
- Usar `np.dot` como producto interno complejo: no conjuga.
- Comparar resultados con `==` en vez de `np.allclose`.
- Asumir que `.T` transpone un vector 1-D: no hace nada.

---
# Ejercicios propuestos

Resuelvelos usando las funciones definidas arriba. Escribe tu propio codigo en
las celdas vacias; la idea es que llegues a la respuesta razonando, no copiando.

1. Sean $u = (2+3i,\, 1-i)$ y $v = (-1+i,\, 4+2i)$. Calcula $3u - (2-i)v$
   y verifica el resultado a mano.
2. Demuestra numericamente que para $c = 1+i$ se cumple
   $\|cv\| = |c|\,\|v\|$ con al menos tres vectores distintos.
3. Construye una matriz $2\times 2$ que sea hermitiana pero **no** unitaria, y
   otra que sea unitaria pero **no** hermitiana. Verificalo con tus funciones.
4. Calcula los valores propios de la matriz de Pauli $Y$ y comprueba que son
   reales (como corresponde a una matriz hermitiana).
5. Verifica que $(AB)^{\dagger} = B^{\dagger}A^{\dagger}$ con matrices
   $3\times 3$ generadas al azar (usa `np.random.rand` para las partes real e
   imaginaria).
6. Comprueba que $\|H \otimes H\|$ aplicado a $|00\rangle$ produce un vector de
   norma 1, y escribe explicitamente cual es ese vector.
7. Dados dos vectores unitarios ortogonales, verifica que su producto interno
   es cero y que la distancia entre ellos es $\sqrt{2}$.
8. Escribe una funcion `es_normal(A)` que verifique si
   $A^{\dagger}A = AA^{\dagger}$, y comprueba que toda matriz hermitiana y toda
   matriz unitaria son normales.

In [ ]:
# Tu solucion aqui

In [ ]:
# Tu solucion aqui

In [ ]:
# Tu solucion aqui

In [ ]:
# Tu solucion aqui

---
# Publicacion en GitHub

Pasos para entregar el cuaderno:

```bash
# 1. Crear el repositorio en github.com (por ejemplo: operaciones-complejas)
# 2. En la carpeta local donde esta el .ipynb:
git init
git add .
git commit -m "Cuaderno de operaciones con vectores y matrices complejas"
git branch -M main
git remote add origin https://github.com/TU_USUARIO/operaciones-complejas.git
git push -u origin main
```

**Antes de subir:**

- Ejecuta `Kernel > Restart & Run All` para que el cuaderno quede con todas las
  salidas visibles y en orden (GitHub renderiza los `.ipynb` directamente).
- Agrega un `README.md` con tu nombre, el curso y una descripcion corta.
- Reemplaza el marcador de nombre al inicio del cuaderno.